# Titanic Survival: A Reproducible Data Workflow

**Name**: Tobias Frei Data **Dataset:** [Titanic - Machine Learning from Disaster](https://www.kaggle.com/competitions/titanic/data) (Kaggle), training split `train.csv` with 891 passengers and 12 columns


**Project Description**: This notebook is a complete workflow that is reproducible. It loads, cleans, explores and visualizes the passanger data from the Titanic. It shows how the survival rate amongst passengers is distributed and investigates how survival rate behaves for different markers like sex, passenger class and age, and how handling of missing ages for passenger changes what the data appears to show. This workflow is intended to serve as a foundation for later machine learning models.

## Table of Content
1. Introduction
2. Data Ingestion
3. Data Familiarization
4. Data Cleaning
5. Exploratory Data Analysis
6. Summary

---

## 1. Introduction
The Titatinc passenger list from Kaggle is a documented record of who did survive the trip on the TITANIC. The iconic phrase "Women and Children First" dates back to this disaster and serves as the prediction to be tested.

### Motivation
I chose this dataset because it is small enough to check every column by hand, yet it has the problems a cleaning workflow has to handle in real data: 
- 136 of the 177 missing ages are in third class
- the cabin column is 3/4 empty
- fare was paid per ticket, not per person. 
That all together makes it a good test case for a workflow in which every cleaning decision is visible and can be repeated and it shows the importance of datawrangling prior to the actual training and building of an ML-model to predict survival rate!



### Research Questions
**Research Question 1**: How did the survival rate differ sex? Where women more likely to survive?

**Research Question 2**: How is age related to survival, did younger passenger show a higher survival rate and how does the way missing age is being handled change the picture?


### Scope
Only the Kaggle training split (train.csv) is used. 
The test split (test.csv) has no Survived column and is deliberately left untouched, so that it remains unseen data for a future model.

---

## 2. Data Ingestion

In [16]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [17]:
df = pd.read_csv('train.csv')

In [18]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [22]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [23]:
df.describe(exclude="number")


,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Braund, Mr. Owen Harris",male,347082,G6,S
freq,1,577,7,4,644


---

## 3. Data Familiarization

In [21]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [24]:
df.isna().mean().mul(100).round(1)

PassengerId     0.0
Survived        0.0
Pclass          0.0
Name            0.0
Sex             0.0
Age            19.9
SibSp           0.0
Parch           0.0
Ticket          0.0
Fare            0.0
Cabin          77.1
Embarked        0.2
dtype: float64

In [25]:
pd.crosstab(df["Pclass"], df["Age"].isna(), margins=True)


Age,False,True,All
Pclass,,,
1,186,30,216
2,173,11,184
3,355,136,491
All,714,177,891


In [27]:
df["Ticket"].value_counts().head()


Ticket
347082      7
1601        7
CA. 2343    7
3101295     6
CA 2144     6
Name: count, dtype: int64

In [28]:
df.loc[df["Ticket"] == "347082", ["Name", "Pclass", "Fare"]]


,Name,Pclass,Fare
13,"Andersson, Mr. Anders Johan",3,31.275
119,"Andersson, Miss. Ellis Anna Maria",3,31.275
541,"Andersson, Miss. Ingeborg Constanzia",3,31.275
542,"Andersson, Miss. Sigrid Elisabeth",3,31.275
610,"Andersson, Mrs. Anders Johan (Alfrida Konstant...",3,31.275
813,"Andersson, Miss. Ebba Iris Alfrida",3,31.275
850,"Andersson, Master. Sigvard Harald Elias",3,31.275


---

## 4. Data Cleansing

---

## 5. Exploratory Data Analysis

---

## 6. Summary